### SILVER LAYER SCRIPT

####Import Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### DATA ACCESS USING APP

In [0]:
bronze_path = "abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/"

display(dbutils.fs.ls(bronze_path))

### DATA LOADING

#### Reading Data

In [0]:
df_cal= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Calendar')

display(df_cal)

In [0]:
df_cus= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Customers')

In [0]:
df_prod= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Products')

In [0]:
df_prod_cat= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Categories')

In [0]:
df_prod_subcat= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Product_Subcategories')

In [0]:
df_ret= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Returns')

In [0]:
df_sales_15= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Sales_2015')

In [0]:
df_sales= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Sales*')

In [0]:
df_sales_17= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Sales_2017')

In [0]:
df_terr= spark.read.format('csv')\
             .option("header", True)\
             .option("inferSchema", True)\
             .load('abfss://bronze@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Territories')

####TRANSFORMATIONS

####Calendar

In [0]:
display(df_cal)

In [0]:
df_cal = df.withColumn('Month', month(col('Date')))\
           .withColumn('Year', year(col('Date')))


In [0]:
df_cal.write.format('parquet')\
            .mode('append')\
            .option("path","abfss://silver@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Calendar")\
            .save()

#####Customers

In [0]:
display(df_cus)

In [0]:
df_cus = df_cus.withColumn("fullname", concat(col('Prefix'), lit(''), col('FirstName'), lit(''), col('LastName'))).display()

In [0]:
df_cus = df_cus.withColumn(
    "fullname",
    concat_ws(" ", col("Prefix"), col("FirstName"), col("LastName"))
)

display(df_cus)


In [0]:
df_cus.write.format('parquet')\
            .mode('append')\
            .option("path","abfss://silver@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Customers")\
            .save()

####Product_Subcategories

In [0]:
display(df_prod_subcat)

In [0]:
df_prod_subcat.write.format('parquet')\
            .mode('append')\
            .option("path","abfss://silver@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Product_Subcategories")\
            .save()

####Products

In [0]:
display(df_prod)

In [0]:
df_prod = df_prod.withColumn('ProductSKU', split(col('ProductSKU'), '-')[0])\
                 .withColumn('ProductName', split(col('ProductName'), ' ')[0])



In [0]:
display(df_prod)

In [0]:
df_prod.write.format('parquet')\
            .mode('append')\
            .option("path","abfss://silver@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Products")\
            .save()

#####Returns

In [0]:
display(df_ret)

In [0]:
df_ret.write.format('parquet')\
            .mode('append')\
            .option("path","abfss://silver@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Returns")\
            .save()

#####Territories

In [0]:
display(df_terr)

In [0]:
df_terr.write.format('parquet')\
            .mode('append')\
            .option("path","abfss://silver@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Territories")\
            .save()

####Product Categories

In [0]:
display(df_prod_cat)

In [0]:
df_prod_cat.write.format('parquet')\
            .mode('append')\
            .option("path","abfss://silver@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Categories")\
            .save()

####Sales

In [0]:
display(df_sales)

In [0]:
df_sales = df_sales.withColumn('StockDate', to_timestamp('StockDate'))

In [0]:
df_sales = df_sales.withColumn('OrderNumber', regexp_replace(col('OrderNumber'), 'S', 'T'))

In [0]:
df_sales = df_sales.withColumn('Multiply', col('OrderLineItem') * col('OrderQuantity'))

In [0]:
display(df_sales)

####Sales Analysis

In [0]:
df_sales.groupBy('OrderDate').agg(count('OrderNumber').alias('Total_Order')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_sales.write.format('parquet')\
            .mode('append')\
            .option("path","abfss://silver@awstoragedatalakethobani.dfs.core.windows.net/AdventureWorks_Sales")\
            .save()